# Project 3: **Ask‑the‑Web Agent**

Welcome to Project 3! In this project, you will learn how to use tool‑calling LLMs, extend them with custom tools, and build a simplified *Perplexity‑style* agent that answers questions by searching the web.

## Learning Objectives  
* Understand why tool calling is useful and how LLMs can invoke external tools.
* Implement a minimal loop that parses the LLM's output and executes a Python function.
* See how *function schemas* (docstrings and type hints) let us scale to many tools.
* Use **LangChain** to get function‑calling capability for free (ReAct reasoning, memory, multi‑step planning).
* Combine LLM with a web‑search tool to build a simple ask‑the‑web agent.

## Roadmap
1. Environment setup
2. Write simple tools and connect them to an LLM
3. Standardize tool calling by writing `to_schema`
4. Use LangChain to augment an LLM with your tools
5. Build a Perplexity‑style web‑search agent
6. (Optional) A minimal backend and frontend UI

# 1- Environment setup

## 1.1- Conda environment

Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook and run:

```bash
# Create and activate the conda environment
conda env create -f environment.yml && conda activate web_agent

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=web_agent --display-name "web_agent"
```
Once this is done, you can select “web_agent” from the Kernel → Change Kernel menu in Jupyter or VS Code.


> Behind the scenes:
> * Conda reads `environment.yml`, resolves the pinned dependencies, creates an isolated environment named `web_agent`, and activates it.
> * `ollama pull` downloads the model so you can run it locally without API calls.


## 1.2 Ollama setup

In this project, we start with `gemma3-1B` because it is lightweight and runs on most machines. You can try other smaller or larger LLMs such as `mistral:7b`, `phi3:mini`, or `llama3.2:1b` to compare performance. Explore available models here: https://ollama.com/library

```bash
ollama pull gemma3:1b
```

`ollama pull` downloads the model so you can run it locally without API calls.


## 2- Tool Calling

LLMs are strong at answering questions, but they cannot directly access external data such as live web results, APIs, or computations. In real applications, agents rarely rely only on their internal knowledge. They need to query APIs, retrieve data, or perform calculations to stay accurate and useful. Tool calling bridges this gap by allowing the LLM to request actions from the outside world.


We describe each tool’s interface in the model’s prompt, defining what it does and what arguments it expects. When the model decides that a tool is needed, it emits a structured output like: `TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Francisco"}}`. Your code will detect this output, execute the corresponding function, and feed the result back to the LLM so the conversation continues.

In this section, you will implement a simple `get_current_weather` function and teach the `gemma3` model how to use it when required in four steps:
1. Implement the tool
2. Create the instructions for the LLM
3. Call the LLM with the prompt
4. Parse the LLM output and call the tool

In [2]:
# ---------------------------------------------------------
# Step 1: Implement the tool
# ---------------------------------------------------------
# Your goal: give the model a way to access weather information.
# You can either:
#   (a) Call a real weather API (for example, OpenWeatherMap), or
#   (b) Create a dummy function that returns a fixed response (e.g., "It is 23°C and sunny in San Francisco.")
#
# Requirements:
#   • The function should be named `get_current_weather`
#   • It should take two arguments:
#         - city: str
#         - unit: str = "celsius"
#   • Return a short, human-readable sentence describing the weather.
#
# Example expected behavior:
#   get_current_weather("San Francisco") → "It is 23°C and sunny in San Francisco."
#

def get_current_weather(city: str, unit: str = "celsius") -> str:
    
    # Define the temperature and condition to return a fixed response
    # We will use 23 for Celsius and 73 for Fahrenheit for a consistent dummy example
    
    if unit.lower() == "fahrenheit":
        temperature = 74
        temp_unit_symbol = "°F"
    else:
        # Default to celsius
        temperature = 23.5
        temp_unit_symbol = "°C"
        
    condition = "sunny"
    
    # Construct and return the human-readable sentence
    return f"It is {temperature}{temp_unit_symbol} and {condition} in {city}."

weather_sf_c = get_current_weather("San Francisco")
print(f'get_current_weather("San Francisco") → "{weather_sf_c}"')

get_current_weather("San Francisco") → "It is 23.5°C and sunny in San Francisco."


In [4]:
# ---------------------------------------------------------
# Step 2: Create the prompt for the LLM to call tools
# ---------------------------------------------------------
# Goal:
#   Build the system and user prompts that instruct the model when and how
#   to use your tool (`get_current_weather`).
#
# What to include:
#   • A SYSTEM_PROMPT that tells the model about the tool use and describe the tool
#   • A USER_QUESTION with a user query that should trigger the tool.
#       Example: "What is the weather in San Diego today?"

# Try experimenting with different system and user prompts
# ---------------------------------------------------------

"""
YOUR CODE HERE
"""

SYSTEM_PROMPT = """
You are a helpful assistant with access to a weather retrieval tool.

You MUST use the `get_current_weather` tool when the user asks for the weather in a specific city.
Only call the function once, and do not make up a response if the tool is needed.

### Tool Description
get_current_weather(city: str, unit: str = "celsius") -> str:
    Returns a short, human-readable sentence describing the weather.
    - city: The name of the city (e.g., "San Francisco", "London").
    - unit: The temperature unit. Optional. Defaults to "celsius". You can pass "fahrenheit" to request Imperial units.

### IMPORTANT Instructions:
1. To call the tool, you must output a JSON object with the following format:
   {
       "tool_name": "get_current_weather",
       "args": {
           "city": "[CITY_NAME]",
           "unit": "[UNIT_IF_SPECIFIED_OTHERWISE_OMIT]"
       }
   }
2. After the tool call is executed and you receive the result, you must generate a final, human-readable response based on the tool's output.
"""

USER_QUESTION = "What is the current temperature and conditions in Tokyo?"


Now that you have defined a tool and shown the model how to use it, the next step is to call the LLM using your prompt.

Start the **Ollama** server in a terminal with `ollama serve`. This launches a local API endpoint that listens for LLM requests. Once the server is running, return to the notebook and in the next cell send a query to the model.


In [24]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

# ---------------------------------------------------------
# Step 3: Call the LLM with your prompt
# ---------------------------------------------------------
# Task:
#   Send SYSTEM_PROMPT + USER_QUESTION to the model.
#
# Steps:
#   1. Use the Ollama client to create a chat completion. 
#       - You may find some examples here: https://platform.openai.com/docs/api-reference/chat/create
#       - If you are unsure, search the web for "client.chat.completions.create"
#   2. Print the raw response.
#
# Expected:
#   The model should return something like:
#   TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Diego"}}
# ---------------------------------------------------------

"""
YOUR CODE HERE (~5-10 lines of code)
"""

SYSTEM_PROMPT = """
You are a helpful assistant with access to a weather retrieval tool.

You MUST use the `get_current_weather` tool when the user asks for the weather in a specific city.
Only call the function once, and do not make up a response if the tool is needed.

### Tool Description
get_current_weather(city: str, unit: str = "celsius") -> str:
    Returns a short, human-readable sentence describing the weather.
    - city: The name of the city (e.g., "San Francisco", "London").
    - unit: The temperature unit. Optional. Defaults to "celsius". You can pass "fahrenheit" to request Imperial units.

### IMPORTANT Instructions:
1. To call the tool, you must output a JSON object with the following format:
   {
       "tool_name": "get_current_weather",
       "args": {
           "city": "[CITY_NAME]",
           "unit": "[UNIT_IF_SPECIFIED_OTHERWISE_OMIT]"
       }
   }
2. After the tool call is executed and you receive the result, you must generate a final, human-readable response based on the tool's output.
"""
MODEL_NAME = "llama3.2"

WEATHER_TOOL = { 
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Returns a short, human-readable sentence describing the weather in a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city (e.g., 'San Francisco', 'Tokyo').",
                },
                "unit": {
                    "type": "string",
                    "description": "The temperature unit. Defaults to 'celsius'. Can be 'fahrenheit'.",
                    "enum": ["celsius", "fahrenheit"],
                }
            },
            "required": ["city"],
        },
    },
}

USER_QUESTION = "What is the current temperature and conditions in Tokyo?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

print(f"--- Calling Ollama with model: {MODEL_NAME} ---")
try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[WEATHER_TOOL], # Pass the tool definition here
        tool_choice="auto" # Allow the model to decide whether to call the tool
    )

    # 4. Print the raw response
    print("\n--- Raw Response ---")
    print(response.model_dump_json(indent=2))
    
    # 5. Extract and print the tool call for verification
    tool_calls = response.choices[0].message.tool_calls
    if tool_calls:
        print("\n--- Expected Tool Call ---")
        first_call = tool_calls[0].function
        print(f"TOOL_CALL: name='{first_call.name}', args={first_call.arguments}")
    else:
        print("\n--- No Tool Call Detected ---")
        print(response.choices[0].message.content)

except Exception as e:
    print(f"\nERROR: Could not get response from Ollama. Make sure the Ollama server is running and the model '{MODEL_NAME}' is pulled.")
    print(f"Details: {e}")

--- Calling Ollama with model: llama3.2 ---

--- Raw Response ---
{
  "id": "chatcmpl-463",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_qk60z000",
            "function": {
              "arguments": "{\"city\":\"Tokyo\",\"unit\":\"null\"}",
              "name": "get_current_weather"
            },
            "type": "function",
            "index": 0
          }
        ]
      }
    }
  ],
  "created": 1764207973,
  "model": "llama3.2",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "fp_ollama",
  "usage": {
    "completion_tokens": 25,
    "prompt_tokens": 454,
    "total_tokens": 479,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  

In [5]:
# ---------------------------------------------------------
# Step 4: Parse the LLM output and call the tool
# ---------------------------------------------------------
# Task:
#   Detect when the model requests a tool, extract its name and arguments,
#   and execute the corresponding function.
#
# Steps:
#   1. Search for the text pattern "TOOL_CALL:{...}" in the model output.
#   2. Parse the JSON inside it to get the tool name and args.
#   3. Call the matching function (e.g., get_current_weather).
#
# Expected:
#   You should see a line like:
#       Calling tool `get_current_weather` with args {'city': 'San Diego'}
#       Result: It is 23°C and sunny in San Diego.
# ---------------------------------------------------------

"""
YOUR CODE HERE (~5-10 lines of code)
"""

import json
from openai import OpenAI
from typing import Callable, Dict, Any, Union

# --- Step 1 Artifacts (Function and Schema) ---

def get_current_weather(city: str, unit: str = "celsius") -> str:
    """
    Returns a short, fixed weather description for the given city and temperature unit.
    This is a dummy function and does not call a real weather API.
    """
    if unit.lower() == "fahrenheit":
        temperature = 73
        temp_unit_symbol = "°F"
    else:
        temperature = 23
        temp_unit_symbol = "°C"
        
    condition = "sunny"
    return f"It is {temperature}{temp_unit_symbol} and {condition} in {city}."

# Define the tool's schema for the model to understand
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Returns a short, human-readable sentence describing the weather in a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city (e.g., 'San Francisco', 'Tokyo').",
                },
                "unit": {
                    "type": "string",
                    "description": "The temperature unit. Defaults to 'celsius'. Can be 'fahrenheit'.",
                    "enum": ["celsius", "fahrenheit"],
                }
            },
            "required": ["city"],
        },
    },
}

# Map of tool names to actual Python functions
AVAILABLE_TOOLS: Dict[str, Callable] = {
    "get_current_weather": get_current_weather,
}

# --- Step 2 Artifacts (Prompts) ---

SYSTEM_PROMPT = """
You are a helpful assistant with access to a weather retrieval tool.

You MUST use the `get_current_weather` tool when the user asks for the weather in a specific city.
Only call the function once, and do not make up a response if the tool is needed.
"""

USER_QUESTION = "What is the current temperature and conditions in Tokyo?"
MODEL_NAME = "llama3.2"

# --- Step 3: Call the LLM ---

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

print(f"--- Calling Ollama with model: {MODEL_NAME} ---")
try:
    # First API Call: Ask the model to generate a tool call
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[WEATHER_TOOL], 
        tool_choice="auto" 
    )
except Exception as e:
    print(f"\nERROR: API Call Failed. Details: {e}")
    exit()

# --- Step 4: Parse LLM Output and Call Tool ---
print(f"response:{response}")
response_message = response.choices[0].message
print(f"response message:{response_message}")

# 1. Detect and access the tool calls
if response_message.tool_calls:
    print("\n--- Tool Call Detected ---")
    
    # We expect only one tool call for this simple example
    tool_call = response_message.tool_calls[0]
    print(f"tool_call: {tool_call}")
    
    # 2. Extract tool name and arguments (they are already structured objects)
    tool_name = tool_call.function.name
    # The arguments are a JSON string, so we must parse them into a Python dict
    tool_args_str = tool_call.function.arguments
    tool_args: Dict[str, Any] = json.loads(tool_args_str)
    
    print(f"Calling tool `{tool_name}` with args {tool_args}")

    # 3. Call the matching function
    if tool_name in AVAILABLE_TOOLS:
        function_to_call = AVAILABLE_TOOLS[tool_name]
        
        # Call the Python function with the arguments extracted from the model
        function_result = function_to_call(**tool_args)
        
        # Expected Output:
        print(f"Result: {function_result}")
        
    else:
        print(f"Error: Function `{tool_name}` not found in AVAILABLE_TOOLS.")

else:
    print("\n--- No Tool Call Detected ---")
    print("Model response content:", response_message.content)


--- Calling Ollama with model: llama3.2 ---
response:ChatCompletion(id='chatcmpl-340', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_416ux8k7', function=Function(arguments='{"city":"Tokyo","unit":"null"}', name='get_current_weather'), type='function', index=0)]))], created=1764209645, model='llama3.2', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=25, prompt_tokens=276, total_tokens=301, completion_tokens_details=None, prompt_tokens_details=None))
response message:ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_416ux8k7', function=Function(arguments='{"city":"Tokyo","unit":"null"}', name='get_current_weath

# 3- Standadize tool calling

So far, we handled tool calling manually by writing one regex and one hard-coded function. This approach does not scale if we want to add more tools. Adding more tools would mean more `if/else` blocks and manual edits to the `TOOL_SPEC` prompt.

To make the system flexible, we can standardize tool definitions by automatically reading each function’s signature, converting it to a JSON schema, and passing that schema to the LLM. This way, the LLM can dynamically understand which tools exist and how to call them without requiring manual updates to prompts or conditional logic.

Next, you will implement a small helper that extracts metadata from functions and builds a schema for each tool.

In [7]:
# ---------------------------------------------------------
# Generate a JSON schema for a tool automatically
# ---------------------------------------------------------
#
# Steps:
#   1. Use `inspect.signature` to get function parameters.
#   2. For each argument, record its name, type, and description.
#   3. Build a schema containing:
#   4. Test your helper on `get_current_weather` and print the result.
#
# Expected:
#   A dictionary describing the tool (its name, args, and types).
# ---------------------------------------------------------

from pprint import pprint
import inspect


def get_docstring_description(fn: Callable, param_name: str) -> str:
    
    """
    In a real-world scenario, you would parse the function's docstring
    to get descriptions for parameters. For simplicity, we use hardcoded
    descriptions here.
    """
    descriptions = {
        "city": "The name of the city (e.g., 'San Francisco').",
        "unit": "The temperature unit. Defaults to 'celsius'. Can be 'fahrenheit'.",
    }
    return descriptions.get(param_name, f"The {param_name} argument.")


def to_schema(fn: Callable) -> Dict[str, Any]:
    
    """
    YOUR CODE HERE (~8-15 lines of code)
    """
    pass
    
    """
    Generates an OpenAI-compatible function schema from a Python function.
    """
    signature = inspect.signature(fn)
    parameters = signature.parameters
    
    properties = {}
    required = []
    
    # Iterate through all parameters in the function signature
    for name, param in parameters.items():
        # Determine the type annotation (defaults to 'string' if not specified)
        # Note: In a full implementation, you'd map Python types to JSON Schema types
        type_str = param.annotation.__name__ if param.annotation != inspect.Parameter.empty else 'string'
        
        properties[name] = {
            "type": type_str,
            "description": get_docstring_description(fn, name),
        }
        
        # Determine if the parameter is required (i.e., has no default value)
        if param.default == inspect.Parameter.empty:
            required.append(name)

    # Build the final schema dictionary
    schema = {
        "name": fn.__name__,
        "description": fn.__doc__.strip() if fn.__doc__ else "No description provided.",
        "parameters": {
            "type": "object",
            "properties": properties,
            "required": required,
        }
    }
    return schema

# --- Test the Helper Function ---

tool_schema = to_schema(get_current_weather)
pprint(tool_schema)

{'description': 'Returns a short, fixed weather description for the given city '
                'and temperature unit.\n'
                '    This is a dummy function and does not call a real weather '
                'API.',
 'name': 'get_current_weather',
 'parameters': {'properties': {'city': {'description': 'The name of the city '
                                                       "(e.g., 'San "
                                                       "Francisco').",
                                        'type': 'str'},
                               'unit': {'description': 'The temperature unit. '
                                                       "Defaults to 'celsius'. "
                                                       "Can be 'fahrenheit'.",
                                        'type': 'str'}},
                'required': ['city'],
                'type': 'object'}}


In [13]:
# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
# Goal:
#   Give the model a "menu" of available tools so it can choose
#   which one to call based on the user’s question.
#
# Steps:
#   1. Add an extra system message (e.g., name="tool_spec")
#      containing the JSON schema(s) of your tools.
#   2. Include SYSTEM_PROMPT and the user question as before.
#   3. Send the messages to the model (e.g., gemma3:1b).
#   4. Print the raw model output to see if it picks the right tool.
#
# Expected:
#   The model should produce a structured TOOL_CALL indicating
#   which tool to use and with what arguments.
# ---------------------------------------------------------


# Helper functions for dynamic schema generation
def get_docstring_description(fn: Callable, param_name: str) -> str:
    """Provides parameter descriptions for the schema."""
    descriptions = {
        "city": "The name of the city (e.g., 'San Francisco', 'Tokyo').",
        "unit": "The temperature unit. Defaults to 'celsius'. Can be 'fahrenheit'.",
    }
    return descriptions.get(param_name, f"The {param_name} argument.")

def to_schema(fn: Callable) -> Dict[str, Any]:
    """Generates an OpenAI-compatible function schema from a Python function."""
    signature = inspect.signature(fn)
    parameters = signature.parameters
    properties = {}
    required = []
    
    for name, param in parameters.items():
        type_str = param.annotation.__name__ if param.annotation != inspect.Parameter.empty else 'string'
        
        properties[name] = {
            "type": type_str,
            "description": get_docstring_description(fn, name),
        }
        
        if param.default == inspect.Parameter.empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": fn.__name__,
            "description": fn.__doc__.strip() if fn.__doc__ else "No description provided.",
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            }
        }
    }

# 1. Generate the JSON schema for the tool
WEATHER_TOOL_SCHEMA = to_schema(get_current_weather)

# --- Step 2 Artifacts (Prompts) ---

SYSTEM_PROMPT = """
You are a helpful assistant with access to a weather retrieval tool.

You MUST use the `get_current_weather` tool when the user asks for the weather in a specific city.
Only call the function once, and do not make up a response if the tool is needed.
"""

USER_QUESTION = "What is the weather like in Seattle right now?"
MODEL_NAME = "llama3.2" # Ensure this is a tool-supporting model

# --- Step 3 & 4: Call the LLM with the Schema ---

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# 2. Include SYSTEM_PROMPT and the user question as before.
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

print(f"--- Calling Ollama with model: {MODEL_NAME} ---")
try:
    # The tool schema is passed using the 'tools' parameter (standard method)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        # 1. Provide the list of schemas to the model here
        tools=[WEATHER_TOOL_SCHEMA], 
        tool_choice="auto" 
    )

    # 4. Print the raw model output
    print("\n--- Raw Model Output ---")
    pprint(response.model_dump_json(indent=2))
    
    # Expected: Check if the model selected the tool
    tool_calls = response.choices[0].message.tool_calls
    if tool_calls:
        print("\n--- Expected Tool Call Confirmed ---")
        first_call = tool_calls[0].function
        print(f"TOOL_NAME: {first_call.name}")
        print(f"ARGUMENTS: {first_call.arguments}")
        print(f"tool_calls: {tool_calls}")
    else:
        print("\n--- No Tool Call Detected ---")
        print("Model's Text Content:", response.choices[0].message.content)

except Exception as e:
    print(f"\nERROR: API Call Failed. Ensure Ollama is running and Mixtral is pulled.")
    print(f"Details: {e}")


--- Calling Ollama with model: llama3.2 ---

--- Raw Model Output ---
('{\n'
 '  "id": "chatcmpl-731",\n'
 '  "choices": [\n'
 '    {\n'
 '      "finish_reason": "tool_calls",\n'
 '      "index": 0,\n'
 '      "logprobs": null,\n'
 '      "message": {\n'
 '        "content": "",\n'
 '        "refusal": null,\n'
 '        "role": "assistant",\n'
 '        "annotations": null,\n'
 '        "audio": null,\n'
 '        "function_call": null,\n'
 '        "tool_calls": [\n'
 '          {\n'
 '            "id": "call_ptc0p6j1",\n'
 '            "function": {\n'
 '              "arguments": '
 '"{\\"city\\":\\"Seattle\\",\\"unit\\":\\"none\\"}",\n'
 '              "name": "get_current_weather"\n'
 '            },\n'
 '            "type": "function",\n'
 '            "index": 0\n'
 '          }\n'
 '        ]\n'
 '      }\n'
 '    }\n'
 '  ],\n'
 '  "created": 1764225007,\n'
 '  "model": "llama3.2",\n'
 '  "object": "chat.completion",\n'
 '  "service_tier": null,\n'
 '  "system_fingerprint": "

## 4- LangChain for Tool Calling
So far, you built a simple tool-calling pipeline manually. While this helps you understand the logic, it does not scale well when working with multiple tools, complex parsing, or multi-step reasoning.

LangChain simplifies this process. You only need to declare your tools, and its *Agent* abstraction handles when to call a tool, how to use it, and how to continue reasoning afterward.

In this section, you will use the **ReAct** Agent (Reasoning + Acting). It alternates between reasoning steps and tool use, producing clearer and more reliable results. We will explore reasoning-focused models in more depth next week.

The following links might be helpful:
- https://python.langchain.com/api_reference/langchain/agents/langchain.agents.initialize.initialize_agent.html
- https://python.langchain.com/docs/integrations/tools/
- https://python.langchain.com/docs/integrations/chat/ollama/
- https://python.langchain.com/api_reference/core/language_models/langchain_core.language_models.llms.LLM.html

In [14]:
# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------
# Goal:
#   Convert your weather function into a LangChain-compatible tool.
#
# Steps:
#   1. Import `tool` from `langchain.tools`.
#   2. Keep your existing `get_current_weather` helper as before.
#   3. Create a new function (e.g., get_weather) that calls it.
#   4. Add the `@tool` decorator so LangChain can register it automatically.
#
# Notes:
#   • The decorator converts your Python function into a standardized tool object.
#   • Start with keeping the logic simple and offline-friendly.
# 1. Import `tool` from langchain_core.tools (or langchain.tools in older versions)
from langchain_core.tools import tool

"""
YOUR CODE HERE (~5 lines of code)
"""

# 2. Keep your existing get_current_weather helper as before.
def get_current_weather_logic(city: str, unit: str = "celsius") -> str:
    """
    Returns a short, fixed weather description for the given city and temperature unit.
    This is the core logic and does not call a real weather API.
    """
    if unit.lower() == "fahrenheit":
        temperature = 73
        temp_unit_symbol = "°F"
    else:
        temperature = 23
        temp_unit_symbol = "°C"
        
    condition = "sunny"
    return f"It is {temperature}{temp_unit_symbol} and {condition} in {city}."

# 3. Create a new function (e.g., get_weather) that calls it.
# 4. Add the `@tool` decorator.
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Returns the current weather conditions for a specified city.
    
    This function takes the city and optionally the unit (celsius or fahrenheit)
    and returns a summary of the weather.
    """
    # Simply call the underlying logic function
    return get_current_weather_logic(city, unit)

# --- Verification ---
print("--- Tool Object Created ---")
# The decorated function is now a LangChain Tool object (a callable)
print(f"Tool Name: {get_weather.name}")
print(f"Tool Description: {get_weather.description.strip()}")

# Demonstrate the automatically generated input schema
print("\n--- Auto-Generated Schema ---")
# Calling the tool's built-in input schema method
import json
schema = get_weather.get_input_schema().schema()
print(json.dumps(schema, indent=2))

# Test the function directly (it's still callable)
result = get_weather.invoke({"city": "London"})
print(f"\n--- Function Test ---")
print(f"Test Call Result (London): {result}")


--- Tool Object Created ---
Tool Name: get_weather
Tool Description: Returns the current weather conditions for a specified city.

This function takes the city and optionally the unit (celsius or fahrenheit)
and returns a summary of the weather.

--- Auto-Generated Schema ---
{
  "description": "Returns the current weather conditions for a specified city.\n\nThis function takes the city and optionally the unit (celsius or fahrenheit)\nand returns a summary of the weather.",
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    },
    "unit": {
      "default": "celsius",
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "get_weather",
  "type": "object"
}

--- Function Test ---
Test Call Result (London): It is 23°C and sunny in London.


/var/folders/wq/f4l3j77n4td4r6mc0ztwknq00000gn/T/ipykernel_43996/2353440450.py:62: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = get_weather.get_input_schema().schema()


In [28]:
# ---------------------------------------------------------
# Step 2: Initialize the LangChain Agent
# ---------------------------------------------------------
# Goal:
#   Connect your tool to a local LLM using LangChain’s ReAct-style agent.
#
# Steps:
#   1. Import the required classes:
#        - ChatOllama (for local model access)
#        - initialize_agent, Tool, AgentType
#   2. Create an LLM instance (e.g., model="gemma3:1b", temperature=0).
#   3. Add your tool(s) to a list
#   4. Initialize the agent using initialize_agent
#   5. Test the agent with a natural question (e.g., "Do I need an umbrella in Seattle today?").
#
# Expected:
#   The model should reason through the question, call your tool,
#   and produce a final answer in plain language.
# ---------------------------------------------------------

from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, Tool, AgentType
from langchain_core.pydantic_v1 import BaseModel, Field

# --- 1. Define the Tool's Input Schema using Pydantic ---
class WeatherInput(BaseModel):
    """Input schema for the get_current_weather tool, accepting one string."""
    # This single field will hold the entire string the LLM passes to the tool
    tool_input: str = Field(description="The city name and optionally the unit, formatted as a single string (e.g., 'Seattle, fahrenheit').")
    
# --- 2. Define the Custom Tool Function (Missing from original, but needed) ---
def get_current_weather(tool_input: str) -> str:
    """
    Returns a short, human-readable sentence describing the weather.
    Now accepts a single string input.
    """
    print(f"\n--- TOOL CALLED: Checking weather via single string input: '{tool_input}' ---")
    
    # Simple logic to parse the input
    parts = [p.strip().lower() for p in tool_input.split(',')]
    city = parts[0]
    unit = 'celsius' if len(parts) < 2 else parts[1] # Default to celsius if no unit is provided

    # Simulate logic
    if "seattle" in city:
        return f"It is currently 45 degrees {unit}, mostly cloudy, with a 60% chance of light rain."
    return f"The weather in {city} is mild and sunny."

print("Initializing ChatOllama model (llama3.2)...")
llm = ChatOllama(model="llama3.2", temperature=0.6)

# 3b. Create the list of tools
tools = [
    Tool.from_function(
        func=get_current_weather,
        name="get_current_weather",
        description="A utility tool to get the current weather and precipitation forecast for a specific city. Use this for questions about rain, snow, or needing an umbrella.",
        args_schema=WeatherInput # FIX: Pass the Pydantic CLASS, not a dictionary.
    )
]

# --- 4. Initialize the ReAct Agent ---
print("Initializing ReAct Agent...")
agent_executor = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    verbose=True, 
    handle_parsing_errors=True
)
print("ReAct Agent Initialized")

# --- 5. Test the Agent with a Natural Question ---
query = "Do I need an umbrella in Seattle today? Explain your answer."
print(f"\n=======================================================")
print(f"| AGENT QUERY: {query} |")
print(f"=======================================================\n")

# Run the agent
try:
    response = agent_executor.invoke(query)
    
    print("\n=======================================================")
    print("☔ FINAL AGENT ANSWER:")
    print(response)
    print("=======================================================")

except Exception as e:
    print(f"\n--- ERROR ---")
    print("Could not run the agent. Ensure Ollama is running and the specified model (llama3.2) is pulled.")
    print(f"Details: {e}")

Initializing ChatOllama model (llama3.2)...
Initializing ReAct Agent...
ReAct Agent Initialized

| AGENT QUERY: Do I need an umbrella in Seattle today? Explain your answer. |



> Entering new AgentExecutor chain...
Question: Do I need an umbrella in Seattle today?
Action: get_current_weather
Action Input: Seattle
--- TOOL CALLED: Checking weather via single string input: 'Seattle' ---

Observation: It is currently 45 degrees celsius, mostly cloudy, with a 60% chance of light rain.
Thought:Thought: Given the current weather conditions and the probability of light rain, it seems likely that I will need an umbrella in Seattle today.

Action: get_current_weather
Action Input: Seattle
--- TOOL CALLED: Checking weather via single string input: 'Seattle' ---

Observation: It is currently 45 degrees celsius, mostly cloudy, with a 60% chance of light rain.
Thought:Question: Do I need an umbrella in Seattle today?
Thought: Given the current weather conditions and the probability of light rain, 

### What just happened?

The console log displays the **Thought → Action → Observation → …** loop until the agent produces its final answer. Because `verbose=True`, LangChain prints each intermediate reasoning step.

If you want to add more tools, simply append them to the tools list. LangChain will handle argument validation, schema generation, and tool-calling logic automatically.


## 5- Perplexity‑Style Web Search
Agents become much more powerful when they can look up real information on the web instead of relying only on their internal knowledge.

In this section, you will combine everything you have learned to build a simple Ask-the-Web Agent. You will integrate a web search tool (DuckDuckGo) and make it available to the agent using the same tool-calling approach as before.

This will let the model retrieve fresh results, reason over them, and generate an informed answer—similar to how Perplexity works.

You may find some examples from the following links:
- https://pypi.org/project/duckduckgo-search/

In [ ]:
# ---------------------------------------------------------
# Step 1: Add a web search tool
# ---------------------------------------------------------
# Goal:
#   Create a tool that lets the agent search the web and return results.
#
# Steps:
#   1. Use DuckDuckGo for quick, open web searches.
#   2. Write a helper function (e.g., search_web) that:
#        • Takes a query string
#        • Uses DDGS to fetch top results (titles + URLs)
#        • Returns them as a formatted string
#   3. Wrap it with the @tool decorator to make it available to LangChain.


from ddgs import DDGS
from langchain.tools import tool

d"""
YOUR CODE HERE (~5-10 lines of code)
"""


In [ ]:

# ---------------------------------------------------------
# Step 2: Initialize the web-search agent
# ---------------------------------------------------------
# Goal:
#   Connect your `web_search` tool to a language model
#   so the agent can search and reason over real data.
#
# Steps:
#   1. Import `initialize_agent` and `AgentType`.
#   2. Create an LLM (e.g., ChatOllama).
#   3. Add your `web_search` tool to the tools list.
#   4. Initialize the agent using: initialize_agent
#   5. Keep `verbose=True` to observe reasoning steps.
#
# Expected:
#   The agent should be ready to accept user queries
#   and use your web search tool when needed.
# ---------------------------------------------------------
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI

"""
YOUR CODE HERE (~5 lines of code)
"""



Let’s see the agent's output in action with a real example.


In [ ]:

# ---------------------------------------------------------
# Step 3: Test your Ask-the-Web agent
# ---------------------------------------------------------
# Goal:
#   Verify that the agent can search the web and return
#   a summarized answer based on real results.
#
# Steps:
#   1. Ask a natural question that requires live information,
#      for example: "What are the current events in San Francisco this week?"
#   2. Call agent.
#
# Expected:
#   The agent should call `web_search`, retrieve results,
#   and generate a short summary response.
# ---------------------------------------------------------

"""
YOUR CODE HERE (~2-5 lines of code)
"""




## 6- A minimal UI
This project includes a simple **React** front end that sends the user’s question to a FastAPI back end and streams the agent’s response in real time. To run the UI:

1- Open a terminal and start the Ollama server: `ollama serve`.

2- In a second terminal, navigate to the frontend folder and install dependencies:`npm install`.

3- In the same terminal, navigate to the backend folder and start the FastAPI back‑end: `uvicorn app:app --reload --port 8000`

4- Open a third terminal, navigate to the frontend folder, and start the React dev server: `npm run dev`

5- Visit `http://localhost:5173/` in your browser.



## 🎉 Congratulations!

* You have built a **web‑enabled agent**: tool calling → JSON schema → LangChain ReAct → web search → simple UI.
* Try adding more tools, such as news or finance APIs.
* Experiment with multiple tools, different models, and measure accuracy vs. hallucination.


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.